In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path(sys.prefix).parent
%cd {PROJECT_ROOT}

/home/younes/younes/Projects/Python/barid_internship


In [89]:
import polars as pl
import altair as alt

import seaborn as sns
import matplotlib.pyplot as plt
# import numpy as np
# import pandas as pd
# import statsmodels.api as sm
# from statsmodels.graphics.tsaplots import plot_acf
# import calendar
# from fpppy.utils import plot_series, plot_series_stacked, plot_diagnostics
# from scipy.stats import pearsonr
# from plotly import express as px
# from pathlib import Path
# from lxml import html
# from itertools import chain

import importlib
import lib
import offline_historique
import read_data
from normalize import normalize_agence

importlib.reload(lib)
importlib.reload(offline_historique)
from offline_historique import parse_historique_from_folder  # noqa: E402

cfg = pl.Config()
cfg.set_tbl_width_chars(10000)
cfg.set_fmt_str_lengths(100)
cfg.set_tbl_cols(-1)
cfg.set_tbl_rows(10)

alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [3]:
def sort_and_complete_operations(
    df: pl.DataFrame,
    starting_states: list = ["depot"],
    end_states: list = ["aexp"],
) -> pl.DataFrame:
    return df.sort(
        pl.col("Heure_Syst_Oper").first().over("id"),
        pl.col("Heure_Syst_Oper"),
    ).filter(
        pl.col("status_code").first().over("id").is_in(starting_states),
        pl.col("status_code").last().over("id").is_in(end_states),
    )

In [4]:
cabs_inter = set(read_data.read_smi_envoisbyproduitintern_many()["codeenvoi_"])

In [84]:
fields = (
    pl.read_parquet("data/cab_dfs/fields.parquet")
    .filter(Etat="V")
    .filter(pl.col("Cab").is_in(cabs_inter))
    .filter(pl.col("Date_depot").ge(pl.date(2023, 1, 1)))
    .cast({"Id": pl.String})
)
operations = (
    pl.read_parquet("data/cab_dfs/operations.parquet").filter(is_valid="V")
    # .sort("cab", "id", "Heure_Syst_Oper")
    .filter(pl.col("cab").is_in(cabs_inter))
    .pipe(
        sort_and_complete_operations,
        end_states=[
            # "liv",
            "aexp",
        ],
    )
    .pipe(normalize_agence)
)
delivery = pl.read_parquet("data/cab_dfs/delivery.parquet").filter(
    pl.col("cab").is_in(cabs_inter)
)
services = pl.read_parquet("data/cab_dfs/services.parquet").filter(
    pl.col("cab").is_in(cabs_inter)
)

In [6]:
print(operations.head())

shape: (5, 15)
┌───────────────┬──────────┬────────────────┬─────────────────────┬──────────────┬─────────────────────────────────────┬────────────┬──────────────────────┬───────────┬───────────┬─────────┬─────────────┬──────────┬─────────────┬───────────────────────┐
│ cab           ┆ id       ┆ Date_operation ┆ Heure_Syst_Oper     ┆ Statut       ┆ Agence                              ┆ Agent_oper ┆ Etat                 ┆ Date_Etat ┆ Agent_maj ┆ ORIGINE ┆ status_code ┆ is_valid ┆ Agence_city ┆ Agence_type           │
│ ---           ┆ ---      ┆ ---            ┆ ---                 ┆ ---          ┆ ---                                 ┆ ---        ┆ ---                  ┆ ---       ┆ ---       ┆ ---     ┆ ---         ┆ ---      ┆ ---         ┆ ---                   │
│ str           ┆ str      ┆ date           ┆ datetime[μs]        ┆ str          ┆ str                                 ┆ i64        ┆ str                  ┆ str       ┆ str       ┆ str     ┆ str         ┆ str      ┆ str    

In [7]:
print(operations.select(pl.all().n_unique()))

shape: (1, 15)
┌────────┬────────┬────────────────┬─────────────────┬────────┬────────┬────────────┬──────┬───────────┬───────────┬─────────┬─────────────┬──────────┬─────────────┬─────────────┐
│ cab    ┆ id     ┆ Date_operation ┆ Heure_Syst_Oper ┆ Statut ┆ Agence ┆ Agent_oper ┆ Etat ┆ Date_Etat ┆ Agent_maj ┆ ORIGINE ┆ status_code ┆ is_valid ┆ Agence_city ┆ Agence_type │
│ ---    ┆ ---    ┆ ---            ┆ ---             ┆ ---    ┆ ---    ┆ ---        ┆ ---  ┆ ---       ┆ ---       ┆ ---     ┆ ---         ┆ ---      ┆ ---         ┆ ---         │
│ u32    ┆ u32    ┆ u32            ┆ u32             ┆ u32    ┆ u32    ┆ u32        ┆ u32  ┆ u32       ┆ u32       ┆ u32     ┆ u32         ┆ u32      ┆ u32         ┆ u32         │
╞════════╪════════╪════════════════╪═════════════════╪════════╪════════╪════════════╪══════╪═══════════╪═══════════╪═════════╪═════════════╪══════════╪═════════════╪═════════════╡
│ 195793 ┆ 195793 ┆ 1078           ┆ 272222          ┆ 26     ┆ 931    ┆ 3435       ┆

In [88]:
# importlib.reload(lib)
lib.plot_ratio_bar_chart(
    operations,
    pl.col("Agence").last().over("id"),
    pl.col("id").unique().len(),
)

alt.Chart(...)

In [ ]:
print(
    operations
    # .sort(
    #     pl.col("Heure_Syst_Oper").first().over("id"), pl.col("Heure_Syst_Oper")
    # )
    .pipe(sort_and_complete_operations, end_states=["liv"])
    # .group_by("id", maintain_order=True).len().mean()
    .group_by("id", maintain_order=True)
    .agg(duration=pl.col("Heure_Syst_Oper").last() - pl.col("Heure_Syst_Oper").first())
    .mean()
)

shape: (1, 2)
┌──────┬─────────────────────────┐
│ id   ┆ duration                │
│ ---  ┆ ---                     │
│ str  ┆ duration[μs]            │
╞══════╪═════════════════════════╡
│ null ┆ 74d 6h 29m 27s 921639µs │
└──────┴─────────────────────────┘


In [10]:
print(
    operations
    # .pipe(complete_operations, end_states=["aexp"])
    .filter(
        pl.col("Date_operation").lt(pl.date(2026, 1, 1)),
    )
    .group_by("id", maintain_order=True)
    .agg(
        first=pl.col("status_code").first(),
        last=pl.col("status_code").last(),
    )
    .group_by("last", maintain_order=True)
    .len()
    .sort("len", descending=True)
    .with_columns(
        cum_ratio=pl.col("len").cum_sum() / pl.col("len").sum(),
    )
)

shape: (12, 3)
┌─────────┬────────┬───────────┐
│ last    ┆ len    ┆ cum_ratio │
│ ---     ┆ ---    ┆ ---       │
│ str     ┆ u32    ┆ f64       │
╞═════════╪════════╪═══════════╡
│ aexp    ┆ 166162 ┆ 0.917723  │
│ liv     ┆ 6803   ┆ 0.955296  │
│ depot   ┆ 2925   ┆ 0.971451  │
│ recpt   ┆ 1704   ┆ 0.980863  │
│ lev     ┆ 1533   ┆ 0.989329  │
│ liv_ret ┆ 1229   ┆ 0.996117  │
│ anoma   ┆ 341    ┆ 0.998001  │
│ affg    ┆ 246    ┆ 0.999359  │
│ nrcl    ┆ 90     ┆ 0.999856  │
│ recg    ┆ 10     ┆ 0.999912  │
│ aff     ┆ 8      ┆ 0.999956  │
│ chrgctr ┆ 8      ┆ 1.0       │
└─────────┴────────┴───────────┘


In [11]:
def map_groups(df, by_cols, fn):
    return (
        df
        # .sort(pl.col("cab").n_unique().over(by_cols))
        .group_by(by_cols, maintain_order=True)
        .map_groups(
            lambda sub_df: sub_df.pipe(fn)
            .with_columns(Destination=pl.lit(sub_df[by_cols][0]))
        )
    )

In [ ]:
importlib.reload(lib)
df = (
    operations[:10000]
    .sort(pl.col("Heure_Syst_Oper").first().over("id"), pl.col("Heure_Syst_Oper"))
    .pipe(
        sort_and_complete_operations,
        end_states=[
            "liv",
            # "aexp",
        ],
    )
    # .pipe(lib.span_by_id, key="id", date_col="Heure_Syst_Oper")
    # .pipe(lib.survival_table, value_col="lead_days", bucket_size=1)
    .pipe(
        lib.filter_categories_by_threshold,
        "Agence_type",
        100,
        pl.col("cab").unique().len(),
    )
    .sort(pl.col("cab").n_unique().over("Agence_type"))
    .pipe(
        lib.map_groups,
        "Agence_type",
        lambda g: g.pipe(
            lib.span_by_id,
            key="id",
            date_col="Heure_Syst_Oper",
            unit="days",
        ).pipe(
            lib.survival_table,
            value_col="lead_days",
            bucket_size=1,
        ),
    )
    # .sort(pl.col("cab").n_unique().over("Destination"))
    # .group_by("Destination", maintain_order=True)
    # .map_groups(
    #     lambda g: g.pipe(lib.span_by_id, key="id", date_col="Heure_Syst_Oper")
    #     .pipe(lib.survival_table, value_col="lead_days", bucket_size=10)
    #     .with_columns(Destination=pl.lit(g["Destination"][0]))
    # )
    # .filter(pl.col("bucket") > 0)
)

chart = (
    alt.Chart(df)
    .mark_line(point=True)
    .encode(
        x=alt.X("bucket:Q", title="Lead Time Bucket (days)").scale(
            # domainMin=1,
            # type="log",
        ),
        tooltip=[
            alt.Tooltip("bucket:Q", title="days"),
            alt.Tooltip("cum_hazard:Q", title="cum_hazard"),
            alt.Tooltip("events:Q", title="events"),
            alt.Tooltip("survival:Q", title="survival"),
        ],
        y=alt.Y(
            # "pdf:Q",
            # "hazard:Q",
            "survival:Q",
            # "expected_remaining_time:Q",
            # "expected_total_time:Q",
        ).scale(
            # type="log",
        ),
        color="Agence_type",
    )
    .properties(
        width=1100, height=400, title="Aggregated Delivery Lead Time Distribution"
    )
)

# lib.add_dropdown_filter(chart, df, "Agence_type")
chart

alt.Chart(...)